In [7]:
import tkinter as tk
from tkinter import messagebox, scrolledtext
import colorsys
import random
import math

class KohonenClusterAnalysis:
    def __init__(self, root, width, height, points_num=200, clusters_num=4, file_name="kohonen_analysis.txt"):
        self.root = root
        self.width = width
        self.height = height
        self.points_num = points_num
        self.clusters_num = clusters_num
        
        self.analysis_iteration = 0
        self.lr = 0.5  # Начальный шаг обучения (learning rate) для сети Кохонена

        # Генерация цветов для отображения кластеров
        self.colors = []
        for color_index in range(self.clusters_num + 1):
            hue = color_index / (clusters_num + 1)
            rgb = colorsys.hsv_to_rgb(hue, 0.8, 0.9)
            r, g, b = int(rgb[0] * 255), int(rgb[1] * 255), int(rgb[2] * 255)
            self.colors.append(f"#{r:02x}{g:02x}{b:02x}")

        self.points = []     # Координаты библиотек (список кортежей)
        self.clusters = []   # Координаты центров округов (список списков для мутабельности)
        self.state = "points"
        
        self.selected_points_num = 0
        self.selected_clusters_num = 0
        
        # Настройка интерфейса
        self.root.title("Вариант 16")
        self.root.geometry(f"{self.width + 350}x{self.height + 220}")  # увеличили ширину под лог

        # Фреймы для холста и лога
        canvas_frame = tk.Frame(root)
        canvas_frame.pack(side=tk.LEFT, padx=5, pady=5)
        log_frame = tk.Frame(root)
        log_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=5, pady=5)

        # Холст
        self.canvas = tk.Canvas(canvas_frame, width=self.width, height=self.height, bg='white')
        self.canvas.pack()
        self.canvas.bind("<Button-1>", self.on_click)

        # Лог-область (ScrolledText) для вывода евклидовых расстояний и координат
        self.log_text = scrolledtext.ScrolledText(log_frame, width=45, height=35, font=("Consolas", 9))
        self.log_text.pack(fill=tk.BOTH, expand=True)
        self.log_text.insert(tk.END, "=== ЛОГ ОБУЧЕНИЯ СЕТИ КОХОНЕНА ===\n")
        self.log_text.insert(tk.END, "Координаты и евклидовы расстояния будут появляться здесь.\n\n")
        
        # Информационная панель
        self.info_label = tk.Label(
            root, 
            text=f"Шаг 1: Накликайте {self.points_num} точек или нажмите кнопку автогенерации",
            font=("Arial", 11), fg="blue"
        )
        self.info_label.pack(pady=5)

        # Кнопка автоматической генерации 200 точек
        self.btn_auto = tk.Button(root, text="Сгенерировать 200 точек случайно", command=self.auto_generate_points)
        self.btn_auto.pack(pady=2)

        self.file_write = open(file_name, 'w', encoding='utf-8')
        self.file_write.write("СЕТЬ КОХОНЕНА\n")

    def auto_generate_points(self):
        if self.state != "points":
            return
        
        self.canvas.delete("all")
        self.points = []
        
        # Генерируем случайные группы точек (имитация округов Москвы)
        centers = [(200, 200), (600, 200), (300, 500), (550, 600)]
        for _ in range(self.points_num):
            base_center = random.choice(centers)
            x = int(base_center[0] + random.normalvariate(0, 60))
            y = int(base_center[1] + random.normalvariate(0, 60))
            # Удерживаем в границах экрана
            x = max(10, min(self.width - 10, x))
            y = max(10, min(self.height - 10, y))
            
            self.points.append((x, y))
            self.draw_point(x, y, "gray")
            
        self.selected_points_num = self.points_num
        self.state = "clusters"
        self.btn_auto.pack_forget() # Скрываем кнопку автогенерации
        
        self.info_label.config(
            text=f"Шаг 2: Выберите кликом {self.clusters_num} начальных центра кластеров (округов)"
        )
        self.log_text.insert(tk.END, "Автоматически сгенерировано {} точек.\n".format(self.points_num))

    def on_click(self, event):
        x, y = event.x, event.y
        if x < 0 or x >= self.width or y < 0 or y >= self.height:
            return
        
        point = (x, y)
        
        # Режим ручного сбора точек
        if self.state == "points":
            if self.selected_points_num < self.points_num:
                if point not in self.points:
                    self.points.append(point)
                    self.selected_points_num += 1
                    self.draw_point(x, y, "gray")
                    
                    remaining = self.points_num - self.selected_points_num
                    if remaining > 0:
                        self.info_label.config(text=f"Выбрано точек: {self.selected_points_num} (Осталось: {remaining})")
                    else:
                        self.state = "clusters"
                        self.btn_auto.pack_forget()
                        self.info_label.config(text=f"Шаг 2: Выберите {self.clusters_num} начальных центров кластеров")
        
        # Режим выбора центров кластеров
        elif self.state == "clusters":
            if self.selected_clusters_num < self.clusters_num:
                if point not in [tuple(c) for c in self.clusters]:
                    self.clusters.append([x, y]) # Сохраняем как список для изменения координат
                    self.selected_clusters_num += 1
                    self.draw_cluster(x, y, "black")
                    
                    remaining = self.clusters_num - self.selected_clusters_num
                    if remaining > 0:
                        self.info_label.config(text=f"Выберите центры округов (Осталось: {remaining})")
                    else:
                        self.state = "analysis"
                        self.info_label.config(text="Точки и центры заданы. Нажмите 'Эпоха обучения Кохонена'")
                        
                        self.file_write.write(f"Исходные точки: {self.points}\n")
                        self.file_write.write(f"Начальные центры: {self.clusters}\n")
                        
                        # Выводим в лог начальные координаты центров
                        self.log_text.insert(tk.END, "\n=== НАЧАЛЬНЫЕ ЦЕНТРЫ КЛАСТЕРОВ ===\n")
                        for i, c in enumerate(self.clusters):
                            self.log_text.insert(tk.END, f"Центр {i+1}: ({c[0]:.2f}, {c[1]:.2f})\n")
                        
                        # Кнопка для запуска эпохи обучения
                        self.btn_epoch = tk.Button(self.root, text="Эпоха обучения Кохонена", command=self.train_kohonen_epoch, bg="#4CAF50", fg="white")
                        self.btn_epoch.pack(pady=10)

    def train_kohonen_epoch(self):
        """Одна эпоха обучения нейросети Кохонена"""
        if self.state not in ["analysis", "training"]:
            return
            
        self.state = "training"
        self.analysis_iteration += 1
        
        self.file_write.write("-" * 50 + f"\nЭпоха обучения №{self.analysis_iteration} (LR = {self.lr:.4f})\n")
        
        # Перемешиваем точки каждую эпоху для лучшего обучения случайной сети
        random.shuffle(self.points)
        
        # Запоминаем старые позиции для проверки сходимости
        old_clusters = [c.copy() for c in self.clusters]
        
        # --- ВЫЧИСЛЕНИЕ СУММАРНОГО РАССТОЯНИЯ ДО НАЧАЛА ЭПОХИ (для динамики) ---
        total_distance_before = 0.0
        for point in self.points:
            distances = [Euclidean_distance(point, cluster) for cluster in self.clusters]
            best_dist = min(distances)
            total_distance_before += best_dist
        avg_dist_before = total_distance_before / len(self.points) if self.points else 0
        
        # Цикл по всем объектам (Обучение "on-line" / Потоковое)
        for point in self.points:
            # 1. Вычисляем Евклидово расстояние до всех нейронов-центров
            distances = [Euclidean_distance(point, cluster) for cluster in self.clusters]
            
            # 2. Определяем индекс нейрона-победителя (минимальное расстояние)
            winner_idx = distances.index(min(distances))
            
            # 3. Модифицируем веса (координаты) только нейрона-победителя
            # Формула: W_new = W_old + LR * (X - W_old)
            self.clusters[winner_idx][0] += self.lr * (point[0] - self.clusters[winner_idx][0])
            self.clusters[winner_idx][1] += self.lr * (point[1] - self.clusters[winner_idx][1])
        
        # --- ВЫЧИСЛЕНИЕ СУММАРНОГО РАССТОЯНИЯ ПОСЛЕ ЭПОХИ ---
        total_distance_after = 0.0
        for point in self.points:
            distances = [Euclidean_distance(point, cluster) for cluster in self.clusters]
            best_dist = min(distances)
            total_distance_after += best_dist
        avg_dist_after = total_distance_after / len(self.points) if self.points else 0
        
        # Уменьшаем шаг обучения (затухание скорости адаптации)
        self.lr *= 0.92
        
        # Перерисовываем экран
        self.canvas.delete("all")
        
        # Отрисовка точек в цвета их итоговых округов-победителей
        for point in self.points:
            distances = [Euclidean_distance(point, cluster) for cluster in self.clusters]
            winner_idx = distances.index(min(distances))
            self.draw_point(point[0], point[1], self.colors[winner_idx + 1])
            
        # Отрисовка центров нейронов (черные квадраты)
        for idx, cluster in enumerate(self.clusters):
            self.draw_cluster(int(cluster[0]), int(cluster[1]), "black")
            self.file_write.write(f"Нейрон-Центр {idx+1}: X={cluster[0]:.2f}, Y={cluster[1]:.2f}\n")
        
        # --- ВЫВОД ИНФОРМАЦИИ В ЛОГ-ОБЛАСТЬ ---
        self.log_text.insert(tk.END, f"\n{'='*50}\n")
        self.log_text.insert(tk.END, f"ЭПОХА {self.analysis_iteration}  |  LR = {self.lr/0.92:.4f} → {self.lr:.4f}\n")
        self.log_text.insert(tk.END, f"Среднее евклидово расстояние до победителя (до эпохи):  {avg_dist_before:.3f}\n")
        self.log_text.insert(tk.END, f"Среднее евклидово расстояние до победителя (после эпохи): {avg_dist_after:.3f}\n")
        self.log_text.insert(tk.END, "Координаты центров после эпохи:\n")
        for i, c in enumerate(self.clusters):
            self.log_text.insert(tk.END, f"  Центр {i+1}: ({c[0]:.2f}, {c[1]:.2f})\n")
        self.log_text.insert(tk.END, f"Максимальный сдвиг центра: {max(Euclidean_distance(old_clusters[i], self.clusters[i]) for i in range(self.clusters_num)):.4f}\n")
        self.log_text.see(tk.END)  # автоматическая прокрутка вниз
            
        self.info_label.config(
            text=f"Эпоха: {self.analysis_iteration} | Скорость обучения (LR): {self.lr:.3f}\nСеть Кохонена адаптирует веса центров..."
        )
        
        # Проверка сходимости (если центры почти не сдвинулись)
        max_shift = max(Euclidean_distance(old_clusters[i], self.clusters[i]) for i in range(self.clusters_num))
        
        if max_shift < 0.3 or self.lr < 0.01:
            self.state = "done"
            self.btn_epoch.config(state=tk.DISABLED)
            self.log_text.insert(tk.END, "\n*** ОБУЧЕНИЕ ЗАВЕРШЕНО ***\n")
            self.log_text.insert(tk.END, "Итоговые центры кластеров:\n")
            for i, c in enumerate(self.clusters):
                self.log_text.insert(tk.END, f"  Центр {i+1}: ({c[0]:.2f}, {c[1]:.2f})\n")
            messagebox.showinfo("Готово", f"Сеть Кохонена успешно обучена!\nВсего эпох: {self.analysis_iteration}")
            self.file_write.close()
        else:
            self.state = "analysis"  # готовы к следующей эпохе

    def draw_point(self, x, y, color):
        radius = 4
        self.canvas.create_oval(x - radius, y - radius, x + radius, y + radius, fill=color, outline="white")

    def draw_cluster(self, x, y, color):
        size = 6
        # Отрисуем треугольником или большим квадратом для наглядности
        self.canvas.create_rectangle(x - size, y - size, x + size, y + size, fill=color, outline="white", width=2)

def Euclidean_distance(point1, point2):
    return math.sqrt((point1[0] - point2[0])**2 + (point1[1] - point2[1])**2)

if __name__ == "__main__":
    window_width = 1200
    window_height = 1200
    
    POINTS_COUNT = 200   
    CLUSTERS_COUNT = 4   
    
    root = tk.Tk()
    app = KohonenClusterAnalysis(root, window_width, window_height, POINTS_COUNT, CLUSTERS_COUNT)
    root.mainloop()